In [1]:
# ======================================================================================
# Ray Data Fair Scheduler — Shared Pool Lock-Stealing & Bottleneck Starvation Repro
# ======================================================================================
#
# Pipeline Topology:
#   instant_source (0s) -> SlowActor (10s) -> SlowerActor (15s) -> fast_sink (0s)
#
# Root Cause Theory:
#   Ray Data's autoscaler evaluates scaling decisions independently for each stage
#   based purely on instant local utilization, completely blind to downstream capacity
#   or overall pipeline flow control.
#
# Scenario Walkthrough:
#   1. `instant_source` populates the queue instantly. All 512 items queue for `SlowActor`.
#   2. Due to the initial 10s task latency, `SlowActor` produces zero output for the first 10s.
#   3. Consequently, the downstream critical bottleneck (`SlowerActor`) sits at 0 active inputs.
#   4. Meanwhile, the fair scheduler loop fires every few ms. It sees `SlowActor` has high local
#      utilization (util >= 1.75) and aggressively ramps it up to 96 actors by t~30s,
#      completely draining the 64-CPU shared pool.
#   5. When `SlowerActor` finally wakes up and yells for resources (util >= 1.75), it passes
#      the local demand check, but the global shared pool is already entirely hijacked by the
#      upstream. As a result, the true bottleneck stage is hard-capped at its reserved
#      allocation only.
#
# Resource Budget Configuration (128 CPUs, reservation_ratio=0.5, 2 ops):
#   - Reserved allocation per operator       = 32 CPUs
#   - Global shared pool allocation          = 64 CPUs
#   - Maximum cap to drain the shared pool   = 32 (reserved) + 64 (shared) = 96 CPUs
#
# Feasible Fair Counterfactual (The Global Optimum):
#   Since `SlowerActor` (15s) is the definitive critical path bottleneck, a global-aware
#   scheduler should allocate the 64 shared CPUs to `SlowerActor` to maximize end-to-end
#   pipeline throughput, while keeping `SlowActor` at its 32 reserved slots to prevent
#   memory backlog.
#     - Optimal Fair Profile: SlowActor = 32 CPUs | SlowerActor = 96 CPUs (Total = 128, feasible)
#     - Actual Bug Profile  : SlowActor = 96 CPUs | SlowerActor = 32 CPUs (Total = 128, starved)
#
# This structural imbalance leads to a permanent 3x capacity loss on the bottleneck stage.
# ======================================================================================

import time, threading, collections
import ray, ray.data
from ray.data import ActorPoolStrategy

print(f"Ray version: {ray.__version__}")


Ray version: 2.56.0


In [2]:
NUM_FILES    = 512
SLOW_DELAY   = 10.0  # SlowActor: 10s/item (fast upstream, like Download)
                      # SlowActor ramps to 96 by t~30s; queue still has >400 items => pool stolen
SLOWER_DELAY = 15.0  # SlowerActor: 15s/item (slow downstream, like Iterate|Extract); true bottleneck

# Budget (128 CPUs, ratio=0.5, 2 ops):
RESERVED   = 32  # CPUs per op
SHARED     = 64  # shared pool
STARVED_AT = RESERVED + SHARED  # SlowActor actors needed to exhaust shared pool = 96

# Fair counterfactual: SlowActor stays at RESERVED, SlowerActor gets RESERVED + SHARED
# Total CPUs = RESERVED + STARVED_AT = 32 + 96 = 128 = budget (physically feasible)
print(f"Optimal Fair Profile: SlowActor={RESERVED} + SlowerActor={STARVED_AT} = {RESERVED+STARVED_AT} CPUs (= budget)")
print(f"Actual Bug Profile  : SlowActor={STARVED_AT} + SlowerActor={RESERVED} = {STARVED_AT+RESERVED} CPUs (= budget)")
print()
print(f"SlowerActor stage capacity — fair ideal: {STARVED_AT} actors / {SLOWER_DELAY:.0f}s = {STARVED_AT/SLOWER_DELAY:.2f} files/sec")
print(f"SlowerActor stage capacity — starved   : {RESERVED} actors / {SLOWER_DELAY:.0f}s = {RESERVED/SLOWER_DELAY:.2f} files/sec")
print(f"=> {STARVED_AT//RESERVED}x capacity loss on the bottleneck stage")


Optimal Fair Profile: SlowActor=32 + SlowerActor=96 = 128 CPUs (= budget)
Actual Bug Profile  : SlowActor=96 + SlowerActor=32 = 128 CPUs (= budget)

SlowerActor stage capacity — fair ideal: 96 actors / 15s = 6.40 files/sec
SlowerActor stage capacity — starved   : 32 actors / 15s = 2.13 files/sec
=> 3x capacity loss on the bottleneck stage


In [3]:
import ray as _ray

@_ray.remote(num_cpus=0)
class _Counter:
    def __init__(self):
        self._alloc       = collections.defaultdict(int)
        self._active      = collections.defaultdict(int)
        self._peak_alloc  = collections.defaultdict(int)
        self._peak_active = collections.defaultdict(int)
    def alloc(self, s):
        self._alloc[s] += 1
        self._peak_alloc[s] = max(self._peak_alloc[s], self._alloc[s])
    def dealloc(self, s):
        self._alloc[s] = max(0, self._alloc[s] - 1)
    def enter(self, s):
        self._active[s] += 1
        self._peak_active[s] = max(self._peak_active[s], self._active[s])
    def exit(self, s):
        self._active[s] = max(0, self._active[s] - 1)
    def snapshot(self):
        return {"alloc": dict(self._alloc), "active": dict(self._active),
                "peak_alloc": dict(self._peak_alloc), "peak_active": dict(self._peak_active)}

def instant_source(batch):
    return {"item_id": list(range(NUM_FILES))}

def fast_sink(batch):
    return batch

class SlowActor:
    def __init__(self):
        _ray.get_actor("_ctr").alloc.remote("slow")
    def __del__(self):
        try: _ray.get_actor("_ctr").dealloc.remote("slow")
        except Exception: pass
    def __call__(self, batch):
        ctr = _ray.get_actor("_ctr")
        ctr.enter.remote("slow")
        try:
            time.sleep(SLOW_DELAY)
            return batch
        finally:
            ctr.exit.remote("slow")

class SlowerActor:
    def __init__(self):
        _ray.get_actor("_ctr").alloc.remote("slower")
    def __del__(self):
        try: _ray.get_actor("_ctr").dealloc.remote("slower")
        except Exception: pass
    def __call__(self, batch):
        ctr = _ray.get_actor("_ctr")
        ctr.enter.remote("slower")
        try:
            time.sleep(SLOWER_DELAY)
            return batch
        finally:
            ctr.exit.remote("slower")


In [4]:
ray.shutdown()
ray.init(num_cpus=128, _temp_dir="/raid/weijiac/ray_tmp")

ctr = _Counter.options(name="_ctr").remote()
snapshots = []
_stop = threading.Event()

def _monitor():
    t0 = time.perf_counter()
    total = ray.cluster_resources().get("CPU", 16)
    while not _stop.is_set():
        snap = ray.get(ctr.snapshot.remote())
        snapshots.append((
            time.perf_counter() - t0,
            snap["alloc"],
            snap["active"],
            total - ray.available_resources().get("CPU", total),
        ))
        _stop.wait(1.0)

t0 = time.perf_counter()
ds = (
    ray.data.from_items([{"seed": 0}])
    .map_batches(instant_source, batch_size=1)
    .repartition(target_num_rows_per_block=1)
    .map_batches(SlowActor,   batch_size=1, num_cpus=1,
                 compute=ActorPoolStrategy(min_size=1, max_size=STARVED_AT))
    .map_batches(SlowerActor, batch_size=1, num_cpus=1,
                 compute=ActorPoolStrategy(min_size=1, max_size=STARVED_AT))
    .map_batches(fast_sink, batch_size=1, num_cpus=0)
)

monitor = threading.Thread(target=_monitor, daemon=True)
monitor.start()
results = ds.take_all()
_stop.set()
monitor.join(timeout=3)

peaks = ray.get(ctr.snapshot.remote())
print(f"Done: {len(results)} items in {time.perf_counter()-t0:.1f}s")
print(f"Peak actors — slow: {peaks['peak_alloc'].get('slow',0)}  slower: {peaks['peak_alloc'].get('slower',0)}")


2026-07-14 23:21:42,982	INFO worker.py:2015 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8269 


2026-07-14 23:21:45,960	INFO streaming_executor.py:193 -- Starting execution of Dataset dataset_5_0. Full logs are in /raid/weijiac/ray_tmp/session_2026-07-14_23-21-28_664742_3010502/logs/ray-data


2026-07-14 23:21:45,961	INFO streaming_executor.py:194 -- Execution plan of Dataset dataset_5_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(instant_source)->StreamingRepartition[num_rows_per_block=1,strict=False]] -> ActorPoolMapOperator[MapBatches(SlowActor)] -> ActorPoolMapOperator[MapBatches(SlowerActor)] -> TaskPoolMapOperator[MapBatches(fast_sink)]


[2026-07-14 23:21:45,994 E 3010502 3010502] core_worker.cc:2149: Actor with class name: 'MapWorker(MapBatches(SlowActor))' and ID: 'ec39eb6e0d98bd03654fdb2701000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost, the actor restart will fail. See https://github.com/ray-project/ray/issues/53727 for more details.
2026-07-14 23:21:46,017	INFO __init__.py:56 -- Progress will be logged because stdout is a non-interactive terminal.


2026-07-14 23:21:46,051	WARNING resource_manager.py:766 -- Cluster resources are not enough to run any task from TaskPoolMapOperator[MapBatches(instant_source)->StreamingRepartition[num_rows_per_block=1,strict=False]]. The job may hang forever unless the cluster scales up.


...

2026-07-14 23:24:06,925	INFO logging_progress.py:231 -- MapBatches(fast_sink): 487/512


2026-07-14 23:24:06,925	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 16.0B object store


2026-07-14 23:24:06,925	INFO logging_progress.py:192 -- ============================================


2026-07-14 23:24:11,214	INFO streaming_executor.py:327 -- ✔️  Dataset dataset_5_0 execution finished in 145.25 seconds


Done: 512 items in 146.1s
Peak actors — slow: 96  slower: 96


In [5]:
print("=== Timeline (1s samples) ===")
print(f"  {'t':>5}  {'slow_alloc':>10} {'slow_active':>11}  "
      f"{'slower_alloc':>12} {'slower_active':>13}  {'cpu':>4}  note")
print("  " + "-"*90)

slower_started = False
for ts, alloc, active, cpu in snapshots:
    sl_a  = alloc.get("slow",   0)
    sl_t  = active.get("slow",  0)
    sr_a  = alloc.get("slower", 0)
    sr_t  = active.get("slower",0)

    stolen     = max(0, sl_a - RESERVED)
    rem_shared = max(0, SHARED - stolen)
    sr_budget  = RESERVED + rem_shared

    note = ""
    if sr_a == 0 and sl_a > 0:
        note = f"<- slower: 0 input  (slow stealing shared, slower budget if started={sr_budget})"
    elif not slower_started and sr_t > 0:
        slower_started = True
        note = f"<- slower FIRST task  slow={sl_a} actors, shared stolen={stolen}, slower budget={sr_budget}"
    elif sl_a >= STARVED_AT and sr_a <= RESERVED:
        note = f"<- STARVED: slower capped at reserved={RESERVED}, shared pool fully stolen by slow upstream"

    print(f"  {ts:5.1f}s  {sl_a:>10d} {sl_t:>11d}  "
          f"{sr_a:>12d} {sr_t:>13d}  {cpu:>3.0f}  {note}")

print()

# Summary
peak_slow = max((s[1].get("slow", 0) for s in snapshots), default=0)
sr_starved = [s[1].get("slower", 0) for s in snapshots if s[1].get("slow", 0) >= STARVED_AT]
peak_sr_starved = max(sr_starved, default=0)

print(f"SlowActor peak: {peak_slow} actors  (shared pool exhausted at >= {STARVED_AT} actors)")
print(f"SlowerActor during shared pool exhaustion: {peak_sr_starved} actors")
print(f"  => capped at reserved={RESERVED} instead of optimal fair allocation={STARVED_AT}")
print(f"  => optimal profile: SlowActor={RESERVED} + SlowerActor={STARVED_AT} = {RESERVED+STARVED_AT} CPUs (= budget, feasible)")
fair_tput   = STARVED_AT / SLOWER_DELAY
actual_tput = max(1, peak_sr_starved) / SLOWER_DELAY
print(f"SlowerActor stage capacity: {fair_tput:.2f} files/sec (optimal fair) vs {actual_tput:.2f} files/sec (starved)"
      f"  => {int(round(fair_tput / actual_tput))}x capacity loss on bottleneck stage")


=== Timeline (1s samples) ===
      t  slow_alloc slow_active  slower_alloc slower_active   cpu  note
  ------------------------------------------------------------------------------------------
    0.2s           0           0             0             0    2  
    1.2s           1           1             1             0    3  
    2.2s           2           2             1             0    4  
    3.2s           3           3             1             0    6  
    4.2s           4           4             1             0    7  
    5.2s           5           5             1             0    8  
    6.2s           6           6             1             0    9  
    7.2s           8           8             1             0   12  
    8.2s          11          11             1             0   15  
    9.3s          13          13             1             0   17  
   10.3s          15          15             1             0   20  
   11.3s          18          18             1           